# Experiment 1 — Study run (Phases 3–4)

Builds the frozen prepared bundle (Phase 3) and runs the validation-only
paired search that locks the 18 recipe winners + 2 ensembles and freezes the
reporting manifest (Phase 4). No test metric is computed here.

Heavy alternative: `uv run python -m cheatdetect.experiments --phase 4`
performs the same 204 fits headlessly; this notebook can then review
`study/artifacts/phase_04_study/candidates.csv` without refitting.

In [ ]:
import pandas as pd
from IPython.display import display

from cheatdetect.experiments.experiment_1 import runner
from cheatdetect.experiments.experiment_1.config import StudyConfig

config = StudyConfig()

## Phase 3 — frozen prepared bundle

`build_study` reads the frozen Phase 1 audit split, verifies the original
training hashes, extracts the paired parent/micro views once, and fits all
nine recipes on original normal training only. The bundle is saved under
`study/artifacts/phase_03_study/`.

In [ ]:
study = runner.build_study(config)
display(
    study.session_report.groupby("partition", sort=False).agg(
        assigned_session_files=("session_file", "size"),
        contributing_session_files=("included", "sum"),
        parent_windows=("parent_windows", "sum"),
    )
)

## Phase 4 — validation-only search and selection lock

Every candidate is trained on original (or original + synthetic) rows with
the frozen preprocessor, scored on combined validation only, and the best
candidate per recipe and augmentation condition wins (first-candidate ties).
Thresholds are tuned on validation; the ensembles reuse the fitted winners.

In [ ]:
run = runner.run_search(study, config)

In [ ]:
winners = pd.DataFrame(
    [
        {
            "id": winner.id,
            "family": winner.family,
            "recipe": winner.recipe,
            "condition": winner.condition,
            "roc_auc": winner.roc_auc,
            "pr_auc": winner.pr_auc,
            "threshold": winner.threshold,
            "fallback_used": winner.threshold_info["fallback_used"],
        }
        for winner in list(run.winners.values()) + list(run.ensembles.values())
    ]
)
display(winners)

In [ ]:
ok = run.candidates[run.candidates["status"] == "ok"]
for family in ("if", "ocsvm", "ae"):
    print(f"=== {family.upper()} — top 5 candidates by validation ROC-AUC ===")
    display(
        ok[ok["family"] == family]
        .sort_values("roc_auc", ascending=False)
        .head(5)
        .reset_index(drop=True)
    )

In [ ]:
print(f"Primary approaches frozen in the manifest: {len(run.manifest['primary'])}")
print(f"Run directory: {run.directory}")